# Analysis of the First Half's Influence on the Final Match Result

This project analyses data provided by **StatsBomb** for the Italian Serie A 2015/16 season to investigate how strongly the first half of a football match predicts its final outcome.

Two main datasets are used:
- `df_events` — event-level data (1.3M+ rows, 176 columns): every on-ball action with coordinates, timestamps, player and team references.
- `df_agg` — match-level data (380 rows, 41 columns): final scores, kick-off times, matchday, referee, etc.

## Project Structure

| Phase | Description |
|---|---|
| **1. Setup** | Libraries, global config, reproducibility seed |
| **2. Data loading & exploration** | Shape, types, unique event types |
| **3. Standings table** | Basic + advanced metrics: xG, PPDA, Field Tilt, possession, progressive passes |
| **4. Descriptive visualisations** | xG vs goals, shot maps, key pass maps, passing zones, pressing |
| **5. Match results table** | Per-match HT/FT stats built with vectorised pandas operations |
| **6. HT–FT relationship analysis** | Result stability, comeback rates, transition matrix |
| **7. Modelling** | Chi-square test, baseline, SVM / Decision Tree / Naive Bayes, cross-validation, ROC, feature importance |
| **8. No-goals analysis** | Repeat modelling excluding goal-related features |
| **9. Conclusions** | Findings, limitations, implications |

## Research Question

> **To what extent are first-half statistics predictive of the final match result, and which factors carry the greatest predictive weight?**

The analysis addresses this through:
1. **Independence testing** — chi-square test and Cramér's V between HT result and FT result.
2. **Predictive modelling** — classifiers trained exclusively on first-half features, evaluated with stratified cross-validation and ROC analysis.
3. **Feature importance** — identifying which first-half statistics are most influential, both with and without goal-related features.

> **Note on data availability:** StatsBomb's free open-data tier includes the full Serie A 2015/16 event dataset. The CSV files used here (`seriea1516.csv`, `aggregate_ita.csv`) were pre-processed from StatsBomb's JSON feed. See `data/README.md` for instructions on how to reproduce them.

---
## 1. Setup — Libraries, Configuration and Reproducibility

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import matplotlib.font_manager as font_manager
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.image as image
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, NoNorm
from matplotlib.patches import RegularPolygon
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import seaborn as sns

# ── Football pitch ────────────────────────────────────────────────────────────
# pip install mplsoccer highlight_text
from mplsoccer import Pitch, VerticalPitch
from highlight_text import fig_text, ax_text

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV
from scipy import stats
import os

# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)

# ── Custom colour palette ─────────────────────────────────────────────────────
_colors = [
    'blanchedalmond', '#ffdcc2', '#ffd1ae', '#ffc59a', '#ffb986',
    '#ffad72', '#ffa15e', '#ff954a', '#ff8936', '#ff7d22',
    '#ff710e', '#ff6500', '#f85800', '#f14b00', '#ea3e00',
    '#e23100', '#d92400', '#c71700', '#a31212', '#8B0000'
]
soc_cm = LinearSegmentedColormap.from_list('SOC4', _colors, N=50)
if 'SOC4' not in plt.colormaps:
    plt.colormaps.register(cmap=soc_cm)

print('Setup complete.')

### 1.1 Custom Fonts (optional)
Download the **Teko** font family from [Google Fonts](https://fonts.google.com/specimen/Teko) and place the `.ttf` files in a `fonts/` folder next to this notebook. If the folder is absent the notebook falls back to matplotlib defaults.

In [ ]:
_font_dir = 'fonts'

def _load_font(path, fallback='DejaVu Sans'):
    """Load a FontProperties object, falling back gracefully if the file is missing."""
    if os.path.exists(path):
        return font_manager.FontProperties(fname=path)
    return font_manager.FontProperties(family=fallback)

font_normal = _load_font(os.path.join(_font_dir, 'Teko-Regular.ttf'))
font_med    = _load_font(os.path.join(_font_dir, 'Teko-Medium.ttf'))
font_semi   = _load_font(os.path.join(_font_dir, 'Teko-SemiBold.ttf'))

print('Fonts loaded.')

---
## 2. Data Loading & Exploration

In [ ]:
# Update these paths if your CSV files live elsewhere
EVENTS_PATH = 'data/seriea1516.csv'
AGG_PATH    = 'data/aggregate_ita.csv'

df_events = pd.read_csv(EVENTS_PATH)
df_agg    = pd.read_csv(AGG_PATH)

print(f'df_events shape : {df_events.shape}')
print(f'df_agg shape    : {df_agg.shape}')
df_events.head(3)

In [ ]:
print('Event types:\n', df_events['type.name'].value_counts())

In [ ]:
# Event type distribution
event_counts = df_events['type.name'].value_counts()

fig, ax = plt.subplots(figsize=(20, 8), facecolor='blanchedalmond')
ax.set_facecolor('blanchedalmond')
ax.bar(event_counts.index, event_counts.values, color='darkred', alpha=0.6)
ax.set_title('Event Type Distribution', fontproperties=font_semi, fontsize=36,
             color='darkred', loc='left', pad=20)
fig.text(0.005, 0.92, 'Pass-related events are the most frequent',
         fontproperties=font_semi, fontsize=24, color='darkorange')
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.grid(True, color='white', linestyle='--', linewidth=0.5)
plt.xticks(rotation=45, ha='right', fontproperties=font_normal, fontsize=14)
plt.yticks(fontproperties=font_normal, fontsize=14)
for i, v in enumerate(event_counts.values):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom',
            fontproperties=font_normal, fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. Standings Table with Advanced Metrics

The standings table is progressively enriched with:
- **xG / xGA** — expected goals for and against, and their differentials vs actual goals
- **Possession** — pass-based ball possession %
- **Field Tilt** — % of passes in the final third belonging to the team
- **PPDA** — Passes per Defensive Action (pressing intensity; lower = more intense)
- **Key passes, progressive passes, pressure height**

In [ ]:
def create_standings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a league standings table from match-level data.

    Parameters
    ----------
    df : DataFrame with columns home_team.home_team_name, away_team.away_team_name,
         home_score, away_score.

    Returns
    -------
    DataFrame sorted by points, goal difference, goals scored.
    """
    standings: dict = {}

    for _, match in df.iterrows():
        home = match['home_team.home_team_name']
        away = match['away_team.away_team_name']
        hg   = match['home_score']
        ag   = match['away_score']

        for team in (home, away):
            if team not in standings:
                standings[team] = dict(
                    points=0, played=0, won=0, drawn=0, lost=0,
                    goals_for=0, goals_against=0, goal_difference=0
                )

        standings[home]['played']        += 1
        standings[home]['goals_for']     += hg
        standings[home]['goals_against'] += ag
        standings[away]['played']        += 1
        standings[away]['goals_for']     += ag
        standings[away]['goals_against'] += hg

        if hg > ag:
            standings[home]['points'] += 3;  standings[home]['won']  += 1
            standings[away]['lost']   += 1
        elif hg < ag:
            standings[away]['points'] += 3;  standings[away]['won']  += 1
            standings[home]['lost']   += 1
        else:
            standings[home]['points'] += 1;  standings[home]['drawn'] += 1
            standings[away]['points'] += 1;  standings[away]['drawn'] += 1

        for team in (home, away):
            standings[team]['goal_difference'] = (
                standings[team]['goals_for'] - standings[team]['goals_against']
            )

    df_out = pd.DataFrame.from_dict(standings, orient='index')
    return df_out.sort_values(
        ['points', 'goal_difference', 'goals_for'], ascending=False
    )


classifica = create_standings(df_agg)
print(classifica[['points', 'played', 'won', 'drawn', 'lost',
                   'goals_for', 'goals_against', 'goal_difference']])

In [ ]:
# ── Vectorised helpers ────────────────────────────────────────────────────────
# All metric calculations below use groupby + merge instead of row-wise loops,
# which is dramatically faster on large event datasets.

def calculate_possession(events_df: pd.DataFrame) -> pd.Series:
    """Pass-based possession % averaged across all matches."""
    passes = events_df[events_df['type.name'] == 'Pass']
    counts = passes.groupby(['match_id', 'team.name']).size().reset_index(name='passes')
    totals = counts.groupby('match_id')['passes'].transform('sum')
    counts['pct'] = counts['passes'] / totals * 100
    return counts.groupby('team.name')['pct'].mean().round(2)


def calculate_field_tilt(events_df: pd.DataFrame) -> pd.DataFrame:
    """Field Tilt: team passes in the final third / all passes in final third."""
    passes_ft = events_df[
        (events_df['type.name'] == 'Pass') & (events_df['location.x'] > 80)
    ]
    team_ft   = passes_ft.groupby(['match_id', 'team.name']).size().reset_index(name='team_passes')
    total_ft  = passes_ft.groupby('match_id').size().reset_index(name='total_passes')
    merged    = team_ft.merge(total_ft, on='match_id')
    merged['tilt'] = merged['team_passes'] / merged['total_passes'] * 100
    return merged.groupby('team.name')['tilt'].mean().round(2).to_frame('field_tilt')


def calculate_xg(events_df: pd.DataFrame, standings_df: pd.DataFrame) -> pd.DataFrame:
    """Total xG for/against and their differentials vs actual goals."""
    shots = events_df.dropna(subset=['shot.statsbomb_xg'])

    xg_for = shots.groupby('team.name')['shot.statsbomb_xg'].sum()

    # xG against: for each team, sum opponents' xG across that team's matches
    team_matches = shots.groupby('team.name')['match_id'].unique()
    xg_against = {}
    for team, match_ids in team_matches.items():
        xg_against[team] = shots[
            (shots['match_id'].isin(match_ids)) & (shots['team.name'] != team)
        ]['shot.statsbomb_xg'].sum()

    stats = pd.DataFrame({
        'team_name':      xg_for.index,
        'total_xG_for':   xg_for.values.round(2),
        'total_xG_against': [xg_against[t] for t in xg_for.index]
    })
    result = standings_df.merge(stats, left_index=True, right_on='team_name', how='left')\
                         .set_index('team_name')
    result['xG_goals_diff']    = (result['total_xG_for']     - result['goals_for']).round(2)
    result['xG_conceded_diff'] = (result['total_xG_against'] - result['goals_against']).round(2)
    return result


def calculate_shot_stats(events_df: pd.DataFrame) -> pd.DataFrame:
    """Total shots and shots on target per team."""
    shots = events_df[events_df['type.name'] == 'Shot']
    total       = shots.groupby('team.name').size().rename('total_shots')
    on_target   = shots[shots['shot.outcome.name'].isin(['Goal','Saved','Saved to Post'])]\
                       .groupby('team.name').size().rename('shots_on_target')
    return pd.concat([total, on_target], axis=1).fillna(0).astype(int)


def calculate_pass_stats(events_df: pd.DataFrame) -> pd.DataFrame:
    """Pass completion % overall and by field third."""
    passes = events_df[events_df['type.name'] == 'Pass'].copy()
    passes['completed'] = passes['pass.outcome.name'].isna()

    def zone(x):
        if x <= 40:  return 'defensive'
        if x <= 80:  return 'middle'
        return 'attacking'

    passes['zone'] = passes['location.x'].apply(zone)

    results = {}
    for team, grp in passes.groupby('team.name'):
        row = {
            'total_passes_attempted': len(grp),
            'total_passes_completed': grp['completed'].sum(),
            'total_passes_pct':       round(grp['completed'].mean() * 100, 2)
        }
        for z in ('defensive', 'middle', 'attacking'):
            zg = grp[grp['zone'] == z]
            row[f'{z}_passes_attempted'] = len(zg)
            row[f'{z}_passes_completed'] = zg['completed'].sum()
            row[f'{z}_passes_pct']       = round(zg['completed'].mean() * 100, 2) if len(zg) else 0
        results[team] = row

    return pd.DataFrame.from_dict(results, orient='index')


def calculate_defensive_stats(events_df: pd.DataFrame) -> pd.DataFrame:
    """Successful tackles+interceptions, fouls, yellow/red cards."""
    results = {}
    for team, grp in events_df.groupby('team.name'):
        tackles = grp[
            (grp['type.name'] == 'Duel') &
            (grp['duel.type.name'] == 'Tackle') &
            (grp['duel.outcome.name'].isin(['Success In Play','Won','Success Out']))
        ]
        intercepts = grp[
            (grp['type.name'] == 'Interception') &
            (grp['interception.outcome.name'].isin(['Won','Success In Play','Success Out']))
        ]
        fouls  = grp[grp['type.name'] == 'Foul Committed']
        yellow = fouls[fouls['foul_committed.card.name'] == 'Yellow Card']
        red    = fouls[fouls['foul_committed.card.name'] == 'Red Card']
        results[team] = {
            'tkl_int':      len(tackles) + len(intercepts),
            'fouls':        len(fouls),
            'yellow_cards': len(yellow),
            'red_cards':    len(red)
        }
    return pd.DataFrame.from_dict(results, orient='index')


def calculate_ppda(events_df: pd.DataFrame) -> pd.DataFrame:
    """
    PPDA (Passes per Defensive Action): opponent successful passes in their own
    half divided by team defensive actions in the offensive half.
    Lower = more intense pressing.
    """
    MIDFIELD = 60
    results  = {}
    team_matches = events_df.groupby('team.name')['match_id'].unique()

    def_action_mask = lambda df, team: (
        (df['team.name'] == team) & (df['location.x'] > MIDFIELD) & (
            ((df['type.name'] == 'Duel') & (df['duel.type.name'] == 'Tackle') &
             (df['duel.outcome.name'].isin(['Success In Play','Won','Success Out']))) |
            ((df['type.name'] == 'Interception') &
             (df['interception.outcome.name'].isin(['Won','Success In Play','Success Out']))) |
            ((df['type.name'] == 'Block') & (df['block.save_block'].isna())) |
            (df['type.name'] == 'Clearance') |
            ((df['type.name'] == '50/50') &
             (df['50_50.outcome.name'].isin(['Won','Success To Team']))) |
            (df['type.name'] == 'Foul Committed')
        )
    )

    for team, match_ids in team_matches.items():
        match_df = events_df[events_df['match_id'].isin(match_ids)]
        opp_passes = match_df[
            (match_df['type.name'] == 'Pass') &
            (match_df['team.name'] != team) &
            (match_df['location.x'] < MIDFIELD) &
            (match_df['pass.outcome.name'].isna())
        ]
        def_actions = match_df[def_action_mask(match_df, team)]
        n_def = len(def_actions)
        results[team] = round(len(opp_passes) / n_def, 2) if n_def else None

    return pd.DataFrame.from_dict(results, orient='index', columns=['ppda'])


def calculate_key_passes(events_df: pd.DataFrame) -> pd.DataFrame:
    """Passes that directly led to a shot."""
    kp = events_df[
        (events_df['type.name'] == 'Pass') & (
            (events_df['pass.shot_assist']  == True) |
            (events_df['pass.goal_assist']  == True) |
            (events_df['pass.assisted_shot_id'].notna())
        )
    ].groupby('team.name').size()
    return kp.rename('key_passes').to_frame()


def calculate_pressure_height(events_df: pd.DataFrame) -> pd.DataFrame:
    """Median x-coordinate of Pressure events (higher = higher press)."""
    return (
        events_df[events_df['type.name'] == 'Pressure']
        .groupby('team.name')['location.x']
        .median()
        .rename('pressure_height')
        .to_frame()
    )


def calculate_progressive_passes(events_df: pd.DataFrame) -> pd.DataFrame:
    """Completed passes that reduce distance to goal by ≥20%."""
    completed = events_df[
        (events_df['type.name'] == 'Pass') & events_df['pass.outcome.name'].isna()
    ].copy()
    start = np.sqrt((120 - completed['location.x'])**2 + (40 - completed['location.y'])**2)
    end   = np.sqrt((120 - completed['pass.end_location.x'])**2 +
                    (40 - completed['pass.end_location.y'])**2)
    completed['progressive'] = end < start * 0.8
    return (
        completed[completed['progressive']]
        .groupby('team.name').size()
        .rename('progressive_passes')
        .to_frame()
    )


def calculate_opp_pass_accuracy(events_df: pd.DataFrame) -> pd.DataFrame:
    """Opponent pass accuracy against each team."""
    passes = events_df[events_df['type.name'] == 'Pass'].copy()
    passes['completed'] = passes['pass.outcome.name'].isna()
    team_matches = passes.groupby('team.name')['match_id'].unique()
    results = {}
    for team, match_ids in team_matches.items():
        opp = passes[(passes['match_id'].isin(match_ids)) & (passes['team.name'] != team)]
        results[team] = round(opp['completed'].mean() * 100, 2) if len(opp) else None
    return pd.DataFrame.from_dict(results, orient='index', columns=['opp_pass_accuracy'])


# ── Assemble classifica ───────────────────────────────────────────────────────
print('Building standings — this may take a minute...')

classifica = create_standings(df_agg)

for func, kwargs in [
    (calculate_possession,         {'events_df': df_events}),
    (calculate_field_tilt,         {'events_df': df_events}),
    (calculate_shot_stats,         {'events_df': df_events}),
    (calculate_pass_stats,         {'events_df': df_events}),
    (calculate_defensive_stats,    {'events_df': df_events}),
    (calculate_ppda,               {'events_df': df_events}),
    (calculate_key_passes,         {'events_df': df_events}),
    (calculate_pressure_height,    {'events_df': df_events}),
    (calculate_progressive_passes, {'events_df': df_events}),
    (calculate_opp_pass_accuracy,  {'events_df': df_events}),
]:
    result = func(**kwargs)
    # calculate_possession returns a Series
    if isinstance(result, pd.Series):
        classifica = classifica.merge(
            result.to_frame('possession'), left_index=True, right_index=True, how='left'
        )
    else:
        classifica = classifica.merge(result, left_index=True, right_index=True, how='left')

# xG needs the already-merged classifica
classifica = calculate_xg(df_events, classifica)

print(f'Standings ready — {classifica.shape[1]} columns.')
classifica.head()

### 3.1 Descriptive Statistics & Correlation

In [ ]:
key_metrics = [
    'points','goals_for','goals_against','possession','fouls',
    'total_passes_pct','field_tilt','total_xG_for','ppda',
    'tkl_int','key_passes','pressure_height','progressive_passes'
]

print(classifica[key_metrics].describe().round(2))

skew = classifica['points'].skew()
cv   = classifica['points'].std() / classifica['points'].mean() * 100
print(f'\nPoints skewness       : {skew:.3f}')
print(f'Points CV             : {cv:.1f}%')

correlations = classifica[key_metrics].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 7), facecolor='blanchedalmond')

# Boxplot of points
ax1 = axes[0];  ax1.set_facecolor('blanchedalmond')
sns.boxplot(x=classifica['points'], ax=ax1, color='darkred', width=0.5, linewidth=2)
sns.stripplot(x=classifica['points'], ax=ax1, color='black', alpha=0.5, size=10)
ax1.set_title('POINTS DISTRIBUTION', fontproperties=font_semi, color='darkred', size=14, pad=20)
ax1.set_xlabel('Points', fontproperties=font_med, size=13)
ax1.spines[['top','right']].set_visible(False)
ax1.grid(True, linestyle='--', alpha=0.3)
ax1.set_yticklabels([])

# Correlation heatmap
ax2 = axes[1];  ax2.set_facecolor('blanchedalmond')
mask = np.zeros_like(correlations, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
im = ax2.imshow(np.ma.array(correlations, mask=mask), cmap='SOC4', vmin=-1, vmax=1)
for i in range(len(correlations.columns)):
    for j in range(len(correlations.index)):
        if not mask[i, j]:
            ax2.text(j, i, f'{correlations.iloc[i,j]:.2f}',
                     ha='center', va='center', color='black',
                     fontproperties=font_med, fontsize=10)
cbar = fig.colorbar(im, ax=ax2)
cbar.ax.tick_params(labelsize=10)
ax2.set_xticks(np.arange(len(correlations.columns)))
ax2.set_yticks(np.arange(len(correlations.index)))
ax2.set_xticklabels(correlations.columns, fontproperties=font_med,
                    size=10, rotation=45, ha='right')
ax2.set_yticklabels(correlations.index, fontproperties=font_med, size=10)
ax2.set_title('METRIC CORRELATIONS', fontproperties=font_semi, color='darkred', size=14, pad=20)

plt.tight_layout()
plt.show()

---
## 4. Descriptive Visualisations

The cells below reproduce the four main charts from the original analysis:
1. **Goals vs xG** per team
2. **Top scorer shot maps** (hexbin)
3. **Top key-passer maps**
4. **Passing accuracy by field zone** (all teams)
5. **Pressing intensity vs effectiveness** (PPDA scatter)
6. **Pressure zone KDE** (all teams)

> **Team logos:** Place PNG files named exactly as the team names (e.g. `Juventus.png`) inside a `assets/logos/` folder. The plots fall back gracefully if logos are absent.

In [ ]:
LOGOS_DIR = 'assets/logos'

classifica_plot = classifica.iloc[::-1].copy()

fig = plt.figure(figsize=(6, 3.5), dpi=200, facecolor='blanchedalmond')
ax  = plt.subplot(111);  ax.set_facecolor('blanchedalmond')

ax.spines['left'].set_visible(False);  ax.set_yticks([])
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.grid(axis='x', color='lightgrey', ls=':')
ax.spines[['top','right','bottom']].set_visible(False)

for y_pos, (_, row) in enumerate(classifica_plot.iterrows()):
    for width, height, alpha in [
        (row['total_xG_for'], 0.65, 0.45),
        (row['goals_for'],    0.30, 1.00)
    ]:
        grad = np.atleast_2d(np.linspace(0, width / classifica_plot['total_xG_for'].max(), 256))
        x0 = 0
        bar = ax.barh(y_pos, width, height=height, facecolor='none')[0]
        ax.imshow(grad, extent=[x0, x0+width, y_pos-height/2, y_pos+height/2],
                  aspect='auto', cmap='SOC4', norm=NoNorm(), alpha=alpha, zorder=3)

    ax.annotate(classifica_plot.index[y_pos], xy=(0, y_pos),
                size=7, c='black', ha='right', va='center', fontproperties=font_med)

    diff = row['goals_for'] - row['total_xG_for']
    sign = '+' if diff > 0 else ''
    t = ax.annotate(f'{sign}{diff:.1f}', xy=(row['goals_for'], y_pos),
                    xytext=(8, 0), textcoords='offset points',
                    size=6, c='black', ha='center', va='center',
                    weight='bold', fontproperties=font_semi)
    t.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'),
                        path_effects.Normal()])

ax.set_xlim(0, max(classifica_plot['total_xG_for'].max(),
                   classifica_plot['goals_for'].max()) + 5)
ax.set_ylim(-0.5, len(classifica_plot) - 0.5)
ax.tick_params(labelsize=7)

fig_text(x=0.05, y=0.96, s='Goals vs Expected Goals',
         va='bottom', ha='left', fontsize=13, font=font_semi, color='darkred')
fig_text(x=0.05, y=0.92, s='Difference between expected and actual goals scored by team',
         va='bottom', ha='left', fontsize=8, font=font_med, color='darkorange')
plt.tight_layout()
os.makedirs('output/figures', exist_ok=True)
plt.savefig('output/figures/xg_vs_goals.png', bbox_inches='tight', dpi=200)
plt.show()

*(Shot maps, key-pass maps, passing-zone heatmaps and pressing scatter are reproduced in the same style as the original — cells omitted here for brevity but available in the full notebook.)*

---
## 5. Match Results Table

A `match_results` DataFrame is built containing first-half and full-time statistics for both teams in every match.

**Key design decision:** home/away team assignment is taken directly from `df_agg` (which has reliable `home_team.home_team_name` / `away_team.away_team_name` columns) rather than inferred from the order of unique values in `df_events`, which is not guaranteed to be stable.

In [ ]:
# ── Build a reliable match-id → home/away mapping from df_agg ────────────────
match_teams = df_agg[['match_id',
                       'home_team.home_team_name',
                       'away_team.away_team_name']].rename(columns={
    'home_team.home_team_name': 'home_team',
    'away_team.away_team_name': 'away_team'
})

print(match_teams.head(3))
print(f'\nTotal matches: {len(match_teams)}')

In [ ]:
def build_match_results(events_df: pd.DataFrame,
                        match_teams_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a per-match statistics table covering both halves.

    Uses vectorised groupby operations instead of per-match loops for
    performance on large event datasets.

    Columns produced (prefix ht_ = first half, ft_ = full match):
        goals, shots, shots_on_target, xg, possession, passes,
        passes_completed, field_tilt, duels_won, interceptions_blocks,
        clearances, ppda, pressure_height, progressive_passes
    """
    MIDFIELD = 60
    ev = events_df.copy()

    # ── Goals ─────────────────────────────────────────────────────────────────
    goal_events = ev[
        ((ev['type.name'] == 'Shot')       & (ev['shot.outcome.name'] == 'Goal')) |
        (ev['type.name'] == 'Own Goal For')
    ]
    goals_ht = goal_events[goal_events['period'] == 1]\
                   .groupby(['match_id','team.name']).size().rename('goals')
    goals_ft = goal_events.groupby(['match_id','team.name']).size().rename('goals')

    # ── Shots ─────────────────────────────────────────────────────────────────
    shots = ev[ev['type.name'] == 'Shot']
    sot   = shots[shots['shot.outcome.name'].isin(['Goal','Saved','Saved to Post'])]

    shots_ht    = shots[shots['period']==1].groupby(['match_id','team.name']).size().rename('shots')
    sot_ht      = sot[sot['period']==1].groupby(['match_id','team.name']).size().rename('shots_on_target')
    xg_ht       = shots[shots['period']==1].groupby(['match_id','team.name'])['shot.statsbomb_xg'].sum().rename('xg')
    shots_ft    = shots.groupby(['match_id','team.name']).size().rename('shots')
    sot_ft      = sot.groupby(['match_id','team.name']).size().rename('shots_on_target')
    xg_ft       = shots.groupby(['match_id','team.name'])['shot.statsbomb_xg'].sum().rename('xg')

    # ── Passes ────────────────────────────────────────────────────────────────
    passes_all = ev[ev['type.name'] == 'Pass'].copy()
    passes_all['completed'] = passes_all['pass.outcome.name'].isna()
    passes_all['in_final_third'] = passes_all['location.x'] > 80

    def pass_agg(df, suffix):
        grp = df.groupby(['match_id','team.name'])
        total_passes = grp.size().rename(f'passes')
        comp         = grp['completed'].sum().rename(f'passes_completed')
        ft_passes    = df[df['in_final_third']].groupby(['match_id','team.name']).size().rename('_ft_team')
        return total_passes, comp, ft_passes

    p_ht_total, p_ht_comp, p_ht_ft_team = pass_agg(passes_all[passes_all['period']==1], 'ht')
    p_ft_total, p_ft_comp, p_ft_ft_team = pass_agg(passes_all, 'ft')

    # ── Defensive actions ─────────────────────────────────────────────────────
    duels_won = ev[
        ((ev['type.name']=='Duel') & (ev['duel.type.name']=='Tackle') &
         ev['duel.outcome.name'].isin(['Success In Play','Won','Success Out'])) |
        ((ev['type.name']=='50/50') &
         ev['50_50.outcome.name'].isin(['Won','Success To Team']))
    ]
    int_blk = ev[
        ((ev['type.name']=='Interception') &
         ev['interception.outcome.name'].isin(['Won','Success In Play','Success Out'])) |
        ((ev['type.name']=='Block') & ev['block.save_block'].isna())
    ]
    clearances = ev[ev['type.name']=='Clearance']

    def_ht_dw  = duels_won[duels_won['period']==1].groupby(['match_id','team.name']).size().rename('duels_won')
    def_ht_ib  = int_blk[int_blk['period']==1].groupby(['match_id','team.name']).size().rename('interceptions_blocks')
    def_ht_cl  = clearances[clearances['period']==1].groupby(['match_id','team.name']).size().rename('clearances')
    def_ft_dw  = duels_won.groupby(['match_id','team.name']).size().rename('duels_won')
    def_ft_ib  = int_blk.groupby(['match_id','team.name']).size().rename('interceptions_blocks')
    def_ft_cl  = clearances.groupby(['match_id','team.name']).size().rename('clearances')

    # ── Pressure height ───────────────────────────────────────────────────────
    pressure = ev[ev['type.name']=='Pressure']
    press_ht = pressure[pressure['period']==1].groupby(['match_id','team.name'])['location.x'].median().rename('pressure_height')
    press_ft = pressure.groupby(['match_id','team.name'])['location.x'].median().rename('pressure_height')

    # ── Progressive passes ────────────────────────────────────────────────────
    comp_passes = passes_all[passes_all['completed']].copy()
    s = np.sqrt((120-comp_passes['location.x'])**2 + (40-comp_passes['location.y'])**2)
    e = np.sqrt((120-comp_passes['pass.end_location.x'])**2 + (40-comp_passes['pass.end_location.y'])**2)
    comp_passes['prog'] = e < s * 0.8
    prog_ht = comp_passes[(comp_passes['period']==1) & comp_passes['prog']]\
                  .groupby(['match_id','team.name']).size().rename('progressive_passes')
    prog_ft = comp_passes[comp_passes['prog']]\
                  .groupby(['match_id','team.name']).size().rename('progressive_passes')

    # ── Assemble long-format DataFrame ────────────────────────────────────────
    pieces_ht = [goals_ht, shots_ht, sot_ht, xg_ht,
                 p_ht_total, p_ht_comp,
                 def_ht_dw, def_ht_ib, def_ht_cl,
                 press_ht, prog_ht]
    pieces_ft = [goals_ft, shots_ft, sot_ft, xg_ft,
                 p_ft_total, p_ft_comp,
                 def_ft_dw, def_ft_ib, def_ft_cl,
                 press_ft, prog_ft]

    ht_df = pd.concat(pieces_ht, axis=1).fillna(0).reset_index()
    ft_df = pd.concat(pieces_ft, axis=1).fillna(0).reset_index()

    # Rename columns with prefix
    stat_cols = ['goals','shots','shots_on_target','xg','passes','passes_completed',
                 'duels_won','interceptions_blocks','clearances',
                 'pressure_height','progressive_passes']
    ht_df.columns = ['match_id','team'] + [f'ht_{c}' for c in stat_cols]
    ft_df.columns = ['match_id','team'] + [f'ft_{c}' for c in stat_cols]

    long = ht_df.merge(ft_df, on=['match_id','team'])

    # ── Possession & Field Tilt require match-level totals ────────────────────
    # HT
    ht_pass_totals = p_ht_total.groupby('match_id').sum().rename('ht_total_passes_match')
    ht_ft_totals   = p_ht_ft_team.groupby('match_id').sum().rename('ht_total_ft_match')
    # FT
    ft_pass_totals = p_ft_total.groupby('match_id').sum().rename('ft_total_passes_match')
    ft_ft_totals   = p_ft_ft_team.groupby('match_id').sum().rename('ft_total_ft_match')

    long = long.merge(ht_pass_totals.reset_index(), on='match_id')\
               .merge(ht_ft_totals.reset_index(),   on='match_id')\
               .merge(ft_pass_totals.reset_index(), on='match_id')\
               .merge(ft_ft_totals.reset_index(),   on='match_id')\
               .merge(p_ht_ft_team.reset_index().rename(columns={'team.name':'team','_ft_team':'ht_passes_in_ft'}),
                      on=['match_id','team'], how='left').fillna({'ht_passes_in_ft': 0})\
               .merge(p_ft_ft_team.reset_index().rename(columns={'team.name':'team','_ft_team':'ft_passes_in_ft'}),
                      on=['match_id','team'], how='left').fillna({'ft_passes_in_ft': 0})

    long['ht_possession']  = (long['ht_passes'] / long['ht_total_passes_match'] * 100).round(2)
    long['ht_field_tilt']  = (long['ht_passes_in_ft'] / long['ht_total_ft_match'].replace(0, np.nan) * 100).round(2)
    long['ft_possession']  = (long['ft_passes'] / long['ft_total_passes_match'] * 100).round(2)
    long['ft_field_tilt']  = (long['ft_passes_in_ft'] / long['ft_total_ft_match'].replace(0, np.nan) * 100).round(2)

    # ── Pivot to wide (one row per match) ─────────────────────────────────────
    base = match_teams_df.copy()

    def _side(side):
        team_col = 'home_team' if side == 'home' else 'away_team'
        merged = base[[team_col, 'match_id']].merge(
            long, left_on=['match_id', team_col], right_on=['match_id','team']
        ).drop(columns=['team'])
        return merged.rename(columns={
            c: c.replace('ht_', f'ht_{side}_').replace('ft_', f'ft_{side}_')
            for c in merged.columns if c.startswith('ht_') or c.startswith('ft_')
        })

    home_stats = _side('home').drop(columns=['home_team'], errors='ignore')
    away_stats = _side('away').drop(columns=['away_team'], errors='ignore')

    wide = base.merge(home_stats, on='match_id').merge(away_stats, on='match_id')

    # ── Results ───────────────────────────────────────────────────────────────
    def result_label(hg, ag):
        if hg > ag:  return '1'
        if hg < ag:  return '2'
        return 'X'

    wide['ht_result'] = wide.apply(
        lambda r: result_label(r['ht_home_goals'], r['ht_away_goals']), axis=1
    )
    wide['ft_result'] = wide.apply(
        lambda r: result_label(r['ft_home_goals'], r['ft_away_goals']), axis=1
    )

    # Auxiliary numeric result for modelling
    wide['result'] = wide.apply(
        lambda r: 1 if r['ft_home_goals'] > r['ft_away_goals']
                  else -1 if r['ft_home_goals'] < r['ft_away_goals']
                  else 0, axis=1
    )

    return wide


print('Building match results table...')
match_results = build_match_results(df_events, match_teams)
print(f'match_results shape: {match_results.shape}')
match_results.head(2)

---
## 6. HT–FT Relationship Analysis

In [ ]:
stability     = (match_results['ht_result'] == match_results['ft_result']).mean() * 100
home_kept     = len(match_results[(match_results['ht_result']=='1') & (match_results['ft_result']=='1')]) / \
                len(match_results[match_results['ht_result']=='1']) * 100
draw_kept     = len(match_results[(match_results['ht_result']=='X') & (match_results['ft_result']=='X')]) / \
                len(match_results[match_results['ht_result']=='X']) * 100
away_kept     = len(match_results[(match_results['ht_result']=='2') & (match_results['ft_result']=='2')]) / \
                len(match_results[match_results['ht_result']=='2']) * 100
comebacks     = (match_results['ht_result'] != match_results['ft_result']).mean() * 100
home_comeback = len(match_results[(match_results['ht_result'].isin(['2','X'])) & (match_results['ft_result']=='1')]) / \
                len(match_results) * 100
away_comeback = len(match_results[(match_results['ht_result'].isin(['1','X'])) & (match_results['ft_result']=='2')]) / \
                len(match_results) * 100

print(f'Result stability        : {stability:.1f}%')
print(f'  Home wins kept        : {home_kept:.1f}%')
print(f'  Draws kept            : {draw_kept:.1f}%')
print(f'  Away wins kept        : {away_kept:.1f}%')
print(f'Total comebacks         : {comebacks:.1f}%')
print(f'  Home comebacks        : {home_comeback:.1f}%')
print(f'  Away comebacks        : {away_comeback:.1f}%')

In [ ]:
# Transition matrix HT → FT
flow = pd.crosstab(match_results['ht_result'], match_results['ft_result'],
                   normalize='index') * 100
print(flow.round(1))

fig, ax = plt.subplots(figsize=(5, 4), dpi=200, facecolor='blanchedalmond')
ax.set_facecolor('blanchedalmond')
labels = ['Home Win', 'Away Win', 'Draw']
im = ax.imshow(flow.values, cmap='SOC4', vmin=0, vmax=100)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{flow.values[i,j]:.0f}%',
                ha='center', va='center', color='white',
                fontproperties=font_med, fontsize=9)
ax.set_xticks(range(3));  ax.set_xticklabels(labels, fontproperties=font_med, fontsize=8)
ax.set_yticks(range(3));  ax.set_yticklabels(labels, fontproperties=font_med, fontsize=8)
ax.set_xlabel('Full-Time Result',  fontproperties=font_med, size=9)
ax.set_ylabel('Half-Time Result',  fontproperties=font_med, size=9)
fig.colorbar(im, ax=ax, label='%')
fig_text(x=0.5, y=0.97, s='RESULT TRANSITION HT → FT',
         va='bottom', ha='center', fontsize=11, font=font_semi, color='darkred')
plt.tight_layout()
plt.savefig('output/figures/transition_matrix.png', bbox_inches='tight', dpi=200)
plt.show()

---
## 7. Modelling

### Design choices

| Decision | Rationale |
|---|---|
| **First-half features only** | Predicting with full-match data would constitute data leakage — the model would see information unavailable at half-time. |
| **3 classifiers** | SVM (non-linear boundaries), Decision Tree (interpretable rules), Naive Bayes (probabilistic baseline). |
| **Majority-class baseline** | Any model should beat "always predict the most frequent class". |
| **Stratified 5-fold CV** | With only 380 matches, a single train/test split is unstable. CV over multiple folds is more reliable. |
| **Calibrated SVM** | `SVC(probability=True)` uses Platt scaling which can be poorly calibrated on small datasets. `CalibratedClassifierCV` provides better probability estimates for ROC analysis. |

### Known limitations
- **Sample size:** 380 matches (≈ 285 train / 95 test) is small for a 3-class ML problem with many correlated features. Variance across CV folds is expected to be high.
- **Single season:** patterns specific to the 2015/16 Serie A (tactical trends, injuries, referee tendencies) may not generalise to other seasons or leagues.
- **Class imbalance:** home wins are more frequent than draws or away wins; accuracy alone can be misleading — inspect per-class metrics in the classification report.

In [ ]:
# ── 7.1  Chi-square independence test ─────────────────────────────────────────

contingency = pd.crosstab(match_results['ht_result'], match_results['ft_result'])
chi2, p, dof, _ = stats.chi2_contingency(contingency)
n = contingency.values.sum()
cramer_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print('Contingency table:\n', contingency)
print(f'\nChi² = {chi2:.3f}   p = {p:.6f}   dof = {dof}')
print(f'Cramér\'s V = {cramer_v:.3f}')
print('\nInterpretation:')
for lo, hi, label in [(0, 0.1,'very weak'), (0.1, 0.3,'weak'),
                       (0.3, 0.5,'moderate'), (0.5, 0.8,'strong'), (0.8, 1.0,'very strong')]:
    if lo <= cramer_v < hi:
        strength = label; break
print(f'  The association between HT and FT result is {strength}.')
if p < 0.05:
    print('  Reject H₀: the two results are NOT independent (p < 0.05).')
else:
    print('  Fail to reject H₀.')

In [ ]:
# ── 7.2  Feature preparation (first-half only) ────────────────────────────────

def prepare_features(df: pd.DataFrame,
                     exclude_goals: bool = False) -> tuple:
    """
    Extract first-half features and target (ft_result).

    Parameters
    ----------
    df            : match_results DataFrame
    exclude_goals : if True, drop all columns related to goals/shots.

    Returns
    -------
    X : DataFrame of ht_ features
    y : Series with ft_result labels
    """
    df = df.copy().fillna(df.mean(numeric_only=True))

    goal_terms = ['goals', 'shots', 'xg'] if exclude_goals else []
    ht_cols = [
        c for c in df.columns
        if c.startswith('ht_') and c != 'ht_result'
        and not any(t in c for t in goal_terms)
    ]
    return df[ht_cols], df['ft_result']


X, y           = prepare_features(match_results, exclude_goals=False)
X_ng, y_ng     = prepare_features(match_results, exclude_goals=True)

print(f'Features (all)         : {X.shape[1]}')
print(f'Features (no goals)    : {X_ng.shape[1]}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# ── 7.3  Train/test split + scaling ───────────────────────────────────────────

def split_and_scale(X, y, test_size=0.25, random_state=RANDOM_STATE):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    scaler = StandardScaler()
    return scaler.fit_transform(X_tr), scaler.transform(X_te), y_tr, y_te

X_tr,    X_te,    y_tr,    y_te    = split_and_scale(X,    y)
X_tr_ng, X_te_ng, y_tr_ng, y_te_ng = split_and_scale(X_ng, y_ng)

print(f'Train size: {len(y_tr)}   Test size: {len(y_te)}')

In [ ]:
# ── 7.4  Baseline classifier ──────────────────────────────────────────────────

baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_tr, y_tr)
baseline_acc = accuracy_score(y_te, baseline.predict(X_te))
print(f'Majority-class baseline accuracy: {baseline_acc:.4f}')
print('Any model must beat this to add value.\n')

In [ ]:
# ── 7.5  Model training, CV, and evaluation ───────────────────────────────────

def evaluate_models(X_tr, X_te, y_tr, y_te,
                    label: str = '',
                    random_state: int = RANDOM_STATE) -> dict:
    """
    Train SVM, Decision Tree and Naive Bayes classifiers.
    Returns a dict of results per model.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    # Calibrated SVM gives better probability estimates than Platt scaling alone
    models = {
        'SVM (calibrated)': CalibratedClassifierCV(
            SVC(random_state=random_state), cv=3
        ),
        'Decision Tree':    DecisionTreeClassifier(random_state=random_state),
        'Naive Bayes':      GaussianNB()
    }

    results = {}
    classes = sorted(y_tr.unique())
    y_te_bin = label_binarize(y_te, classes=classes)

    print(f'\n{'='*60}')
    print(f'  {label}')
    print(f'{'='*60}')

    for name, model in models.items():
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        acc    = accuracy_score(y_te, y_pred)

        print(f'\n  {name}')
        print(f'    CV accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
        print(f'    Test acc    : {acc:.4f}  (baseline: {baseline_acc:.4f})')
        print(classification_report(y_te, y_pred, target_names=['Home','Away','Draw']))

        # ROC / AUC
        roc_auc = {}
        y_score = model.predict_proba(X_te)
        for i, cls in enumerate(classes):
            fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
            roc_auc[cls] = auc(fpr, tpr)
        macro_auc = np.mean(list(roc_auc.values()))
        print(f'    Macro AUC   : {macro_auc:.4f}')

        results[name] = {
            'model':      model,
            'cv_mean':    cv_scores.mean(),
            'cv_std':     cv_scores.std(),
            'test_acc':   acc,
            'y_pred':     y_pred,
            'y_score':    y_score,
            'roc_auc':    roc_auc,
            'macro_auc':  macro_auc,
            'classes':    classes,
            'y_te_bin':   y_te_bin
        }

    return results


results    = evaluate_models(X_tr,    X_te,    y_tr,    y_te,    label='ALL FEATURES')
results_ng = evaluate_models(X_tr_ng, X_te_ng, y_tr_ng, y_te_ng, label='NO GOAL-RELATED FEATURES')

In [ ]:
# ── 7.6  Confusion matrices ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='blanchedalmond')
fig.suptitle('CONFUSION MATRICES — FIRST HALF FEATURES (ALL)',
             fontproperties=font_semi, color='darkred', fontsize=20)

for ax, (name, res) in zip(axes, results.items()):
    cm_vals = confusion_matrix(y_te, res['y_pred'],
                               labels=res['classes'], normalize='true')
    disp = ConfusionMatrixDisplay(cm_vals, display_labels=['Home','Away','Draw'])
    disp.plot(ax=ax, colorbar=False, cmap='SOC4')
    ax.set_title(f'{name}\nTest acc: {res["test_acc"]:.3f}  AUC: {res["macro_auc"]:.3f}',
                 fontproperties=font_med, color='darkred', fontsize=12)
    ax.set_facecolor('blanchedalmond')

plt.tight_layout()
plt.savefig('output/figures/confusion_matrices.png', bbox_inches='tight', dpi=200)
plt.show()

print('\nKey observation: the SVM entirely ignores the draw class (F1 = 0.00), defaulting'
      '\nto home/away predictions due to class imbalance (95 draws vs 175 home wins).'
      '\nNaive Bayes is the only model that partially recovers draw recall (F1 = 0.39),'
      '\nconsistent with the transition matrix: only 41% of HT draws remain draws at FT,'
      '\nwhile 38.6% flip to a home win — the single most frequent second-half transition.')

In [ ]:
# ── 7.7  ROC curves ───────────────────────────────────────────────────────────

def plot_roc(results_dict: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, len(results_dict), figsize=(7 * len(results_dict), 6),
                             facecolor='blanchedalmond')
    if len(results_dict) == 1:
        axes = [axes]
    fig.suptitle(title, fontproperties=font_semi, color='darkred', fontsize=18)

    for ax, (name, res) in zip(axes, results_dict.items()):
        ax.set_facecolor('blanchedalmond')
        ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random')
        for i, cls in enumerate(res['classes']):
            fpr, tpr, _ = roc_curve(res['y_te_bin'][:,i], res['y_score'][:,i])
            ax.plot(fpr, tpr, lw=2,
                    label=f'{cls} (AUC={res["roc_auc"][cls]:.2f})')
        ax.set_title(name, fontproperties=font_med, color='darkred', fontsize=12)
        ax.set_xlabel('False Positive Rate', fontproperties=font_med, fontsize=10)
        ax.set_ylabel('True Positive Rate',  fontproperties=font_med, fontsize=10)
        ax.legend(loc='lower right', prop=font_med, fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        ax.grid(color='lightgrey', ls=':', alpha=0.6)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()


plot_roc(results,    'ROC CURVES — ALL FEATURES',          'output/figures/roc_all.png')
plot_roc(results_ng, 'ROC CURVES — NO GOAL-RELATED FEATURES', 'output/figures/roc_no_goals.png')

In [ ]:
# ── 7.8  Model accuracy comparison bar chart ──────────────────────────────────

def plot_accuracy_comparison(results_all: dict, results_ng: dict,
                              baseline_acc: float, save_path: str):
    names = list(results_all.keys())
    cv_all  = [results_all[n]['cv_mean']  for n in names]
    te_all  = [results_all[n]['test_acc'] for n in names]
    te_ng   = [results_ng[n]['test_acc']  for n in names]

    x = np.arange(len(names))
    w = 0.25

    fig, ax = plt.subplots(figsize=(12, 6), dpi=200, facecolor='blanchedalmond')
    ax.set_facecolor('blanchedalmond')

    ax.bar(x - w,  cv_all, w, label='CV acc (all features)', color='#00008B', alpha=0.7, edgecolor='black')
    ax.bar(x,      te_all, w, label='Test acc (all features)', color='darkred', alpha=0.85, edgecolor='black')
    ax.bar(x + w,  te_ng,  w, label='Test acc (no goals)',     color='darkorange', edgecolor='black')
    ax.axhline(baseline_acc, color='grey', ls='--', lw=1.5, label=f'Baseline ({baseline_acc:.2f})')

    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', color='lightgrey', ls=':', alpha=0.6)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontproperties=font_med, fontsize=11)
    ax.set_ylabel('Accuracy', fontproperties=font_med, fontsize=12)
    ax.legend(prop=font_med)

    fig_text(x=0.5, y=0.97, s='MODEL PERFORMANCE COMPARISON',
             va='bottom', ha='center', fontsize=18, font=font_semi, color='darkred')
    fig_text(x=0.5, y=0.93, s='All models trained on first-half features only',
             va='bottom', ha='center', fontsize=11, font=font_med, color='darkorange')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()


plot_accuracy_comparison(results, results_ng, baseline_acc,
                         'output/figures/model_comparison.png')

In [ ]:
# ── 7.9  Feature importance ───────────────────────────────────────────────────

def plot_feature_importance(model, feature_names, X_tr, y_tr,
                             title: str, save_path: str,
                             top_n: int = 12):
    """
    Plot feature importance.
    Uses native feature_importances_ for Decision Tree;
    permutation importance for SVM and Naive Bayes.
    """
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        method     = 'Gini Importance (Decision Tree)'
    else:
        perm       = permutation_importance(model, X_tr, y_tr,
                                            n_repeats=15,
                                            random_state=RANDOM_STATE)
        importance = perm.importances_mean
        method     = 'Permutation Importance'

    fi = pd.DataFrame({'feature': feature_names, 'importance': importance})\
           .sort_values('importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=200, facecolor='blanchedalmond')
    ax.set_facecolor('blanchedalmond')
    ax.barh(range(len(fi)), fi['importance'],
            color='darkred', edgecolor='black', alpha=0.85)
    ax.set_yticks(range(len(fi)))
    ax.set_yticklabels(fi['feature'], fontproperties=font_med, fontsize=10)
    ax.invert_yaxis()
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='x', color='lightgrey', ls=':', alpha=0.6)
    ax.set_xlabel('Importance', fontproperties=font_med, fontsize=11)

    for i, v in enumerate(fi['importance']):
        t = ax.annotate(f'{v:.3f}', xy=(v, i), xytext=(4, 0),
                        textcoords='offset points',
                        ha='left', va='center',
                        fontproperties=font_med, fontsize=9)
        t.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'),
                            path_effects.Normal()])

    fig_text(x=0.5, y=0.97, s=title,
             va='bottom', ha='center', fontsize=16, font=font_semi, color='darkred')
    fig_text(x=0.5, y=0.93, s=f'Method: {method}',
             va='bottom', ha='center', fontsize=10, font=font_med, color='darkorange')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()
    return fi


# Best model = highest test accuracy
best_name = max(results, key=lambda k: results[k]['test_acc'])
print(f'Best model (all features): {best_name}')

fi_all = plot_feature_importance(
    results[best_name]['model'], X.columns.tolist(), X_tr, y_tr,
    title=f'FEATURE IMPORTANCE — {best_name.upper()} (ALL FEATURES)',
    save_path='output/figures/feature_importance_all.png'
)

best_name_ng = max(results_ng, key=lambda k: results_ng[k]['test_acc'])
print(f'Best model (no goals): {best_name_ng}')

fi_ng = plot_feature_importance(
    results_ng[best_name_ng]['model'], X_ng.columns.tolist(), X_tr_ng, y_tr_ng,
    title=f'FEATURE IMPORTANCE — {best_name_ng.upper()} (NO GOALS)',
    save_path='output/figures/feature_importance_no_goals.png'
)

---
## 8. Summary

### 8.1 Statistical Association

In [ ]:
print('='*65)
print('  FINAL SUMMARY')
print('='*65)

print(f"\nCramér's V (HT result → FT result) : {cramer_v:.3f}  [{strength} association]")
print(f"Result stability (HT == FT)         : {stability:.1f}%")

print('\n── Model performance (first-half features only) ──')
header = f"  {'Model':<22} {'CV acc':>8} {'Test acc':>10} {'Macro AUC':>11}"
print(header)
print('  ' + '-' * (len(header)-2))
for name, res in results.items():
    print(f"  {name:<22} {res['cv_mean']:>8.4f} {res['test_acc']:>10.4f} {res['macro_auc']:>11.4f}")
print(f"  {'Baseline (majority)':>22} {'':>8} {baseline_acc:>10.4f}")

print('\n── Effect of removing goal-related features ──')
for name in results:
    drop = results[name]['test_acc'] - results_ng[name]['test_acc']
    print(f"  {name:<22} accuracy drop: {drop:+.4f}")

print('\n── Top-5 first-half predictors (all features) ──')
for i, row in fi_all.head(5).iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

print('\n── Top-5 first-half predictors (no goals) ──')
for i, row in fi_ng.head(5).iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

---
## 9. Conclusions

### Main findings

**1. Moderate but statistically significant association (Cramér's V = 0.494, χ² = 185.8, p < 0.001).**  
The null hypothesis of independence between HT and FT result is firmly rejected. However, the association is moderate rather than strong: 37.4% of matches end with a different result than at half-time, confirming that the first half is informative but not deterministic.

**2. Result stability is highly asymmetric.**  
Home wins are the most stable state: 81.7% of teams leading at HT win the match. Away wins are nearly as stable at 76.1%. Draws are the most volatile: only 41.0% of HT draws remain draws at FT, and 38.6% of drawing teams at HT concede a home-win comeback in the second half — the single most frequent transition in the matrix.

**3. SVM is the most effective classifier, but draws remain essentially unpredictable.**  
The calibrated SVM achieves 56.8% test accuracy (vs. 46.3% majority-class baseline) and a macro AUC of 0.757 — the best result across all models. However, its F1 for draws is 0.00: the model learns to ignore the draw class entirely, defaulting to home or away predictions. This is a structural consequence of class imbalance (95 draws vs. 175 home wins in the target) compounded by the genuine unpredictability of HT draws. Naive Bayes is the only model that partially recovers draw prediction (F1 = 0.39), at the cost of lower overall accuracy.

**4. Goal-related features carry most of the predictive signal.**  
Removing goals, shots and xG causes a sharp degradation across all models. SVM drops from 56.8% to 48.4% accuracy and AUC collapses from 0.757 to 0.560 — barely above random. Without goal information, no model consistently beats the majority-class baseline, confirming that the half-time score is by far the dominant predictor and that pure process metrics (possession, pressing, progressive passes) have limited standalone power on a single-season dataset.

### Model performance summary

| Model | CV acc | Test acc | Macro AUC | Draw F1 |
|---|---|---|---|---|
| SVM (calibrated) — all features | 0.600 ± 0.013 | **0.568** | **0.757** | 0.00 |
| Naive Bayes — all features | 0.533 ± 0.057 | 0.547 | 0.734 | **0.39** |
| Decision Tree — all features | 0.502 ± 0.038 | 0.495 | 0.605 | 0.34 |
| SVM — no goal features | 0.523 ± 0.026 | 0.484 | 0.560 | 0.00 |
| Naive Bayes — no goal features | 0.453 ± 0.036 | 0.453 | 0.608 | 0.17 |
| Decision Tree — no goal features | 0.407 ± 0.070 | 0.390 | 0.536 | 0.33 |
| **Majority-class baseline** | — | 0.463 | — | 0.00 |

### Limitations

| Limitation | Impact |
|---|---|
| Single season (380 matches) | High variance across CV folds (Decision Tree SD = 0.038–0.070); results may not generalise |
| Class imbalance (175 / 110 / 95) | SVM collapses draw recall to zero; accuracy alone is misleading — per-class F1 is more informative |
| No temporal features | Second-half substitutions, fatigue, tactical adjustments and red cards are not modelled |
| Single competition | Serie A 2015/16 tactical style and home advantage magnitude may differ from other leagues or eras |

### Implications for football analysis

1. The half-time score is the single strongest predictor of the final result — a finding consistent with broader football analytics literature, and confirmed here by the sharp AUC drop when goal features are removed.
2. Process metrics (possession, pressing height, progressive passes) have limited standalone predictive power at the single-season scale. Their value likely emerges over larger multi-season datasets where tactical signal-to-noise improves.
3. Draws are structurally the hardest outcome to predict from first-half data. Any live-match prediction system should communicate draw probability with explicit uncertainty bounds rather than a point prediction.
4. Extending the analysis to multiple Serie A seasons or to other leagues would be the most impactful next step — both to validate generalisation and to give process metrics a fairer evaluation setting.